# Phase 2 — LangGraph Agent (dual path + retry)

Goal: wrap ingestion + NL2SQL from notebook 01 as LangGraph nodes, add the pandas path,
and test the retry-on-error edge in isolation before this logic moves into `backend/graph.py`.

Run notebook 01 first so `data.db` exists.

In [ ]:
import sqlite3
import pandas as pd
from groq import Groq
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

client = Groq()
MODEL  = 'llama-3.3-70b-versatile'
DB_PATH = 'data.db'
TABLE   = 'data'

def call_llm(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL, max_tokens=512,
        messages=[{'role': 'user', 'content': prompt}],
    )
    return resp.choices[0].message.content.strip()

## State

In [ ]:
class AgentState(TypedDict):
    question:    str
    schema:      str
    db_path:     str
    df:          Optional[pd.DataFrame]
    path:        str
    sql:         Optional[str]
    pandas_code: Optional[str]
    result:      Optional[object]
    error:       str
    retries:     int
    answer:      Optional[str]
    table:       Optional[list]

## Nodes

In [ ]:
PANDAS_KEYWORDS = [
    'correlation','distribution','describe','summary','statistics',
    'std','mean','median','missing','null','shape',
    'how many rows','how many columns','columns','dtypes','head',
]

def router_node(state):
    q = state['question'].lower()
    path = 'pandas' if any(kw in q for kw in PANDAS_KEYWORDS) else 'sql'
    return {**state, 'path': path}

def pick_path(state):
    return state.get('path', 'sql')

In [ ]:
def generate_sql(state):
    hint = f'\nPrevious error: {state["error"]}\nFix it.' if state['error'] else ''
    prompt = (
        'You are a SQLite expert. Write ONE SELECT query. '
        'Return ONLY raw SQL, no markdown.\n\n'
        + state['schema'] + '\n\nQuestion: ' + state['question'] + hint + '\nSQL:'
    )
    sql = call_llm(prompt).strip().strip('`').removeprefix('sql').strip()
    return {**state, 'sql': sql, 'error': ''}

def execute_sql(state):
    try:
        conn = sqlite3.connect(state['db_path'])
        df   = pd.read_sql_query(state['sql'], conn)
        conn.close()
        return {**state, 'result': df.head(100), 'error': ''}
    except Exception as e:
        return {**state, 'result': None, 'error': str(e), 'retries': state['retries']+1}

In [ ]:
def generate_pandas(state):
    prompt = (
        'You are a pandas expert. Write ONE pandas expression. '
        'df is already loaded. Return ONLY the expression, no imports, no markdown.\n\n'
        + state['schema'] + '\n\nQuestion: ' + state['question'] + '\nExpression:'
    )
    code = call_llm(prompt).strip().strip('`').removeprefix('python').strip()
    return {**state, 'pandas_code': code, 'error': ''}

def execute_pandas(state):
    import numpy as np
    sandbox = {'df': state['df'], 'pd': pd, 'np': np, 'len': len, 'round': round}
    try:
        result = eval(state['pandas_code'], {'__builtins__': {}}, sandbox)
        return {**state, 'result': result, 'error': ''}
    except Exception as e:
        return {**state, 'result': None, 'error': str(e)}

In [ ]:
def format_answer(state):
    result = state['result']
    if result is None:
        return {**state, 'answer': 'Could not answer that question.', 'table': None}
    result_str = result.to_string(index=False) if isinstance(result, pd.DataFrame) else str(result)
    table      = result.to_dict(orient='records') if isinstance(result, pd.DataFrame) else None
    prompt = (
        'Write a short 1-2 sentence plain English answer. Include actual numbers.\n\n'
        f'Question: {state["question"]}\nResult:\n{result_str}\n\nAnswer:'
    )
    return {**state, 'answer': call_llm(prompt), 'table': table}

## Build the graph

In [ ]:
def should_retry(state):
    return 'retry' if state['error'] and state['retries'] < 2 else 'format'

g = StateGraph(AgentState)
g.add_node('router',          router_node)
g.add_node('generate_sql',    generate_sql)
g.add_node('execute_sql',     execute_sql)
g.add_node('generate_pandas', generate_pandas)
g.add_node('execute_pandas',  execute_pandas)
g.add_node('format_answer',   format_answer)

g.set_entry_point('router')
g.add_conditional_edges('router', pick_path, {'sql':'generate_sql','pandas':'generate_pandas'})
g.add_edge('generate_sql', 'execute_sql')
g.add_conditional_edges('execute_sql', should_retry, {'retry':'generate_sql','format':'format_answer'})
g.add_edge('generate_pandas', 'execute_pandas')
g.add_edge('execute_pandas',  'format_answer')
g.add_edge('format_answer',   END)

agent = g.compile()
print('Graph compiled OK')

## Load schema + df from notebook 01's database

In [ ]:
conn   = sqlite3.connect(DB_PATH)
cols   = conn.execute(f'PRAGMA table_info({TABLE})').fetchall()
sample = conn.execute(f'SELECT * FROM {TABLE} LIMIT 3').fetchall()
conn.close()

schema = (
    f'Table: {TABLE}\nColumns:\n'
    + '\n'.join(f'  - {c[1]} ({c[2]})' for c in cols)
    + '\nSample rows:\n'
    + '\n'.join(str(r) for r in sample)
)

df = pd.read_sql(f'SELECT * FROM {TABLE}', sqlite3.connect(DB_PATH))
print(schema)

## Run test questions

In [ ]:
questions = [
    'What is the total revenue by region?',          # sql path
    'Which customer spent the most?',                # sql path
    'Show me the correlation between quantity and price',  # pandas path
    'How many rows are there?',                      # pandas path
]

initial = {
    'schema': schema, 'db_path': DB_PATH, 'df': df,
    'path': '', 'sql': None, 'pandas_code': None,
    'result': None, 'error': '', 'retries': 0, 'answer': None, 'table': None,
}

for q in questions:
    print(f'\nQ: {q}')
    out = agent.invoke({**initial, 'question': q})
    print(f'Path  : {out["path"]}')
    print(f'Answer: {out["answer"]}')
    if out.get('sql'):         print(f'SQL   : {out["sql"]}')
    if out.get('pandas_code'): print(f'Pandas: {out["pandas_code"]}')
    print('-' * 60)

## Next: phase 3 (schema RAG)

Once all questions above answer correctly, the agent is solid.
Next notebook tests embedding column metadata into ChromaDB
so large datasets with many columns work without stuffing everything into the prompt.